# Codenames spymaster trainen met GRPO

Dit notebook traint **Qwen2.5-1.5B-Instruct** (LoRA) met reinforcement learning (GRPO) tot betere
Codenames-spymaster. De guesser is **Llama 3.3 70B via Inworld** — dezelfde als in de baseline.

**Voor je start**
1. *Runtime → Change runtime type → T4 GPU* (gratis).
2. Klik links op het 🔑-icoon (Secrets) → *Add new secret*: naam `INWORLD_API_KEY`, waarde = je key, en zet *Notebook access* aan.
3. *Runtime → Run all*. Google Drive vraagt om toestemming: alles wordt daar opgeslagen, zodat niets verloren gaat als Colab de verbinding verbreekt.

**Wat er gebeurt**
1. Setup: code van GitHub, packages, Drive, API-key
2. Model laden + snelle meting vóór training
3. Trainen (± 1–1,5 uur op een T4, ± $0,50–1 aan guesser-calls)
4. Meting ná training + LoRA-adapter opslaan op Drive
5. Exporteren naar GGUF (Q4_K_M, zelfde compressie als Ollama) voor de echte vergelijking op je laptop

Valt de verbinding weg? Gewoon opnieuw *Run all*: de training gaat verder vanaf het laatste checkpoint op Drive.

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/casruger/code-names-trained-llm"

import os
if not os.path.exists("/content/code-names-trained-llm"):
    !git clone -q {REPO_URL} /content/code-names-trained-llm
else:
    !git -C /content/code-names-trained-llm pull -q
%cd /content/code-names-trained-llm
!pip install -q trl peft datasets openai python-dotenv

In [ ]:
# ---- instellingen ---------------------------------------------------------------
RUN_NAME        = "grpo-v1"        # nieuwe naam = nieuwe training (eigen map op Drive)
MODEL_ID        = "Qwen/Qwen2.5-1.5B-Instruct"   # zelfde model als qwen2.5:1.5b in Ollama
GUESSER_MODEL   = "deepinfra/meta-llama/Llama-3.3-70B-Instruct-Turbo"  # NIET wijzigen (baseline)

N_TRAIN_BOARDS  = 3000     # trainingsborden (seeds 0..2999; evaluatieborden beginnen bij 1.000.000)
MAX_STEPS       = 300      # optimizer-stappen; elke stap = 4 borden x 8 hints
NUM_GENERATIONS = 8        # hoeveel hints per bord GRPO met elkaar vergelijkt
BATCH_SIZE      = 8        # hints per forward pass (moet deelbaar zijn door NUM_GENERATIONS)
GRAD_ACCUM      = 4        # -> 8 x 4 = 32 hints = 4 borden per stap
LEARNING_RATE   = 1e-5
LORA_R          = 16
BETA            = 0.04     # KL-straf: houdt het model dicht bij het origineel (0 = vrij)
TEMPERATURE     = 1.0      # sampling tijdens training: variatie is nodig om te kunnen vergelijken
EVAL_BOARDS     = 50       # zelfde 50 evaluatieborden als baseline.py
# ---------------------------------------------------------------------------------

from google.colab import drive, userdata
drive.mount("/content/drive")
OUT = f"/content/drive/MyDrive/codenames-rl/{RUN_NAME}"
os.makedirs(OUT, exist_ok=True)
os.environ["INWORLD_API_KEY"] = userdata.get("INWORLD_API_KEY")
print("Alles wordt opgeslagen in", OUT)

In [ ]:
# Test de guesser (1 API-call)
from codenames import generate_board
from codenames.players import LLMClient, LLMGuesser

guesser = LLMGuesser(LLMClient(GUESSER_MODEL, "inworld", temperature=0.0))
b = generate_board(seed=42)
team = b.hidden_by_role("team")[:2]
print("teamwoorden:", team)
print("guesser zegt:", guesser.guess(b, "test", 2), "| ruw:", guesser.last_raw)

## 2. Model laden en meten vóór training

Deze meting gebruikt dezelfde borden, prompts en scoring als `baseline.py`, maar met het model in
Hugging Face-formaat (16-bit). De score kan iets afwijken van je Ollama-baseline (−2,16, 4-bit);
de officiële vergelijking doe je straks lokaal. Hier dient het als nulmeting voor dít notebook.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()   # T4: nee -> fp16
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "GEEN GPU!", "| dtype:", DTYPE)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE, device_map="cuda")

In [ ]:
import json
from codenames.training import evaluate_hf, CachedGuesser

eval_guesser = CachedGuesser(guesser)
before, before_rows = evaluate_hf(model, tokenizer, eval_guesser, n=EVAL_BOARDS)
json.dump({"summary": before, "rows": before_rows}, open(f"{OUT}/eval_before.json", "w"))
for k, v in before.items(): print(f"{k:<20}{v:8.2f}")

## 3. Trainen met GRPO

Per stap: het model schrijft 8 hints per bord → de guesser raadt → `score_clue_text` geeft de reward →
hints die beter zijn dan het groepsgemiddelde worden waarschijnlijker gemaakt.

Let tijdens het trainen op:
- **`reward`** — gemiddelde reward van de hints; hoort langzaam te stijgen.
- **`frac_reward_zero_std`** — aandeel borden waar alle 8 hints precies even goed scoorden. Daar valt
  niets te vergelijken, dus niets te leren. Blijft dit dicht bij 1, dan leert het model niet.
- **`kl`** — hoe ver het model van het origineel is afgedreven. Langzaam oplopen is normaal.

Alle samples (hint, gokken, reward) komen in `reward_log.jsonl` op Drive.

In [ ]:
from peft import LoraConfig
from trl import GRPOConfig, GRPOTrainer
from codenames.training import build_dataset, CodenamesReward

train_dataset = build_dataset(N_TRAIN_BOARDS, seed=0)
reward_fn = CodenamesReward(guesser, max_workers=16, log_path=f"{OUT}/reward_log.jsonl")

lora = LoraConfig(
    r=LORA_R, lora_alpha=2 * LORA_R, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
config = GRPOConfig(
    output_dir=f"{OUT}/checkpoints",
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_generations=NUM_GENERATIONS,
    max_completion_length=32,          # 'CLUE: x / NUMBER: n' is kort; langer = verspilde tijd
    temperature=TEMPERATURE,
    beta=BETA,
    max_steps=MAX_STEPS,
    logging_steps=5,
    save_steps=25,                     # checkpoint op Drive -> verder kunnen na disconnect
    save_total_limit=2,
    bf16=USE_BF16, fp16=not USE_BF16,
    gradient_checkpointing=True,
    report_to="none",
    seed=0,
)
trainer = GRPOTrainer(
    model=model,
    reward_funcs=[reward_fn],
    args=config,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    peft_config=lora,
)
trainer.model.print_trainable_parameters()

In [ ]:
import glob
resume = bool(glob.glob(f"{OUT}/checkpoints/checkpoint-*"))
print("Verder vanaf checkpoint" if resume else "Nieuwe training")
trainer.train(resume_from_checkpoint=resume)
print(f"guesser-cache: {reward_fn.guesser.hits} hits, {reward_fn.guesser.misses} API-calls")

In [ ]:
# Hoe ging het? Reward per stap + wat voor hints het model is gaan geven.
import collections
import matplotlib.pyplot as plt
import pandas as pd
from codenames.training import read_reward_log

hist = pd.DataFrame([h for h in trainer.state.log_history if "reward" in h])
log = pd.DataFrame(read_reward_log(f"{OUT}/reward_log.jsonl"))
log["valid"] = log["error"].isna()
w = max(1, len(log) // 40)

fig, ax = plt.subplots(1, 4, figsize=(20, 4))
ax[0].plot(hist["step"], hist["reward"]); ax[0].set_title("gem. reward per stap")
ax[1].plot(log["valid"].rolling(w * 32).mean().values); ax[1].set_title("aandeel geldige hints")
ax[2].plot(log["n_correct"].rolling(w * 32).mean().values); ax[2].set_title("gem. goed geraden per hint")
ax[3].plot(hist["step"], hist["frac_reward_zero_std"]); ax[3].set_title("borden zonder leersignaal\n(alle 8 hints even goed)")
for a in ax: a.grid(alpha=.3)
plt.show()

last = log.tail(min(1000, len(log)))
print("Meest gegeven hints in de laatste 1000 samples (let op reward hacking!):")
print(collections.Counter(last.loc[last.valid, "clue"]).most_common(15))
print("\nFouten:", collections.Counter(e.split("'")[0] for e in last.error.dropna()).most_common(5))

## 4. Meten ná training en opslaan

In [ ]:
after, after_rows = evaluate_hf(trainer.model, tokenizer, eval_guesser, n=EVAL_BOARDS)
json.dump({"summary": after, "rows": after_rows}, open(f"{OUT}/eval_after.json", "w"))
print(f"{'metric':<20}{'voor':>8}{'na':>8}")
for k in before: print(f"{k:<20}{before[k]:8.2f}{after[k]:8.2f}")
print("\nVoorbeelden na training:")
for r in after_rows[:8]: print(f"  {r['clue']!s:>12} {r['number']!s:>2} | {r['reward']:+.1f} | {r['error'] or r['outcomes']}")

In [ ]:
# De LoRA-adapter (~20-40 MB) = wat het model geleerd heeft
ADAPTER_DIR = f"{OUT}/lora_adapter"
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
json.dump({k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str))},
          open(f"{OUT}/config.json", "w"), indent=2)
print("Adapter opgeslagen in", ADAPTER_DIR)

## 5. Exporteren naar GGUF voor Ollama

Adapter + basismodel worden samengevoegd en gecomprimeerd naar **Q4_K_M** — dezelfde compressie als
`qwen2.5:1.5b` in Ollama, zodat de vergelijking met je baseline eerlijk is. Duurt ± 5–10 minuten.

In [ ]:
merged = trainer.model.merge_and_unload()
merged.save_pretrained("/content/merged", safe_serialization=True)
tokenizer.save_pretrained("/content/merged")

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q sentencepiece gguf
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outfile /content/spymaster-f16.gguf --outtype f16
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/spymaster-f16.gguf /content/codenames-spymaster-q4_k_m.gguf Q4_K_M
!cp /content/codenames-spymaster-q4_k_m.gguf "{OUT}/"
!ls -lh "{OUT}"

## 6. Op je laptop: importeren in Ollama en vergelijken

1. Download `codenames-spymaster-q4_k_m.gguf` uit Google Drive (`codenames-rl/grpo-v1/`) naar je projectmap.
2. In PowerShell, in je projectmap — kopieer de instellingen van het originele model en wissel alleen de gewichten:
   ```powershell
   ollama show qwen2.5:1.5b --modelfile > Modelfile
   (Get-Content Modelfile) -replace '^FROM .*', 'FROM ./codenames-spymaster-q4_k_m.gguf' | Set-Content Modelfile
   ollama create codenames-spymaster -f Modelfile
   ```
3. Dezelfde meting als de baseline:
   ```powershell
   python baseline.py --model codenames-spymaster --guesser-provider inworld --guesser-model deepinfra/meta-llama/Llama-3.3-70B-Instruct-Turbo --runs 5
   ```
4. Vergelijk `eval_results/codenames-spymaster__vs__.../summary.json` met de baseline (mean reward **−2,16**, range −2,61 … −1,60).